 <span style="font-size:20px"> **AI for Cell Image Analysis** <a id=0> </a> </span>

 <span style="font-size:30px"> lab2plus_intro2DL-cpu  (optional) </span>

 _Student name:_<font color = 'blue'> (田昊阳)</font> <br>
_Student ID:_<font color = 'blue'> (2400013518)</font> <br>

# <span style="font-size:30px"> Lab2. Introduction to Deep Learning (Advanced) </span>

**Author: [Zehua Zhao 赵泽华 zzh@stu.pku.edu.cn](mailto:zzh@stu.pku.edu.cn)**   <br>
*Assisted by: Claude Opus 4* <br>
Instructional texts added and slightly modified by: Xiao Li

This notebook is designed for students with zero background in deep learning, progressively introducing core **deep learning"** concepts through five classic dataset-network combinations: 
- advancing to: color image classification with CNN+CIFAR-10
- advancing to: deep network training with ResNet+Fashion-MNIST
- advanced application: image segmentation with U-Net+Oxford Pets
- advanced application:modifying the U-Net with ResNet blocks for enhanced performance.

---

## <span style="font-size:25px"> Task 0: Environment Setup </span>

Installing necessary Python libraries

In [1]:
'''
#No need to install if run on Bohrium
# Install PyTorch and other dependencies for image processing
%pip install torch torchvision torchaudio
# If you have GPU support, you can try the GPU version by uncommenting the next line.
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # For CUDA 11.8 support
%pip install pillow
%pip install tqdm
'''

'\n#No need to install if run on Bohrium\n# Install PyTorch and other dependencies for image processing\n%pip install torch torchvision torchaudio\n# If you have GPU support, you can try the GPU version by uncommenting the next line.\n# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # For CUDA 11.8 support\n%pip install pillow\n%pip install tqdm\n'

---

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm #tqdm is a Python library for progress bar
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import PIL
from PIL import Image

This part automatically detect if you have GPU available. The program will use  GPU if available, otherwise will use CPU.

In [3]:
# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


---

## Task 2: CIFAR-10 Classification with CNN

`CIFAR-10` (Canadian Institute For Advanced Research) is a color image dataset containing 10 classes, with 6,000 32×32 pixel RGB images per class. There are 60,000 images in total, with 50,000 for training and 10,000 for testing. The 10 classes include: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck. While MNIST dataset deals with grayscale images, most images in everyday life and research are color images. Fluorescent cell images are also mostly color images. CIFAR is more challenging than MNIST and is commonly used to test the performance of color image classification algorithms.

 

### 2.1 Imports

In [4]:
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


### 2.2 Load Data

In [5]:
# Data preprocessing
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load datasets
print("Downloading CIFAR-10 dataset...")
train_dataset = torchvision.datasets.CIFAR10(root='/share/image_cloud/data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='/share/image_cloud/data', train=False, transform=transform_test)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

# CIFAR-10 classes
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Function to denormalize images for visualization
def denormalize(tensor):
    mean = torch.tensor([0.4914, 0.4822, 0.4465])
    std = torch.tensor([0.2023, 0.1994, 0.2010])
    return tensor * std.view(3, 1, 1) + mean.view(3, 1, 1)

URLError: <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)>

In [ ]:
# TODO: Visualize CIFAR-10 samples
# Hint: Create a 2x5 grid showing one sample from each class

# TODO end

plt.suptitle('CIFAR-10 Sample Visualization')
plt.tight_layout()
plt.show()

In [ ]:
print("\nCIFAR-10 Dataset Information:")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Number of classes: 10")

### 2.3 Define CNN Model

`Convolutional Neural Network (CNN)` is a deep learning architecture specifically designed for processing data with grid-like topology, such as images. The core components of CNN include convolutional layers, pooling layers, and fully connected layers. **Convolutional layers** extract local features through convolution kernels, **pooling layers** reduce feature dimensions while maintaining translation invariance, and **fully connected layers** perform final classification or regression. CNN's local connectivity and weight sharing characteristics significantly reduce the number of parameters, leading to revolutionary success in computer vision tasks.

<img src="https://bohrium.oss-cn-zhangjiakou.aliyuncs.com/article/12769/f2203fb02d4342b9bc72ee09a4394813/9bedf1b6-973d-4384-b0bb-2d8a962d3ac5.png" alt="CNN" style="width: 800px"/> </br>
**Figure 2**. A convolutional neural network (CNN) archetecture for image recognition.




Figure 2 illustrates a typical CNN archetecture for image recognition. </br>
Let's go through the codes for defining a simple CNN model. </br>
Again, the number of features for the input and output layers are fixed.

- _<font color='purple'>input layer</font>_</br>
    size: 3x32x32 </br> 
    Note: Color image has 3 channels (R,G,B).
- _<font color='purple'>output layer</font>_</br>
    Number of output features = 10 </br>
    The CNN classifier should outputs one of the 10 categories: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck
- _<font color='purple'>convolutional layers</font>_ </br>
    At the heart of a CNN is the **convolution operation**, which involves sliding a small filter (or **kernel**) over the input image. This process allows the network to learn **local patterns** such as edges, textures, and corners.
    - A **kernel** is a small matrix (k*k), often 3×3 or 5×5 in size.
    - It "scans" the image from top-left to bottom-right.
    - At each position, it performs an element-wise multiplication with the image patch and sums the result to produce a **feature map** value.

    A convolutional layer can be conveniently built by callying PyTorch function `nn.Conv2d()`. </br>
    `nn.Conv2d`(_in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None_)</br>
    The first confolutional layer can be set as:</br>
    <font face="Courier New">self.conv1 = nn.Conv2d(3, <font color='blue'>cn1</font>, <font color='blue'>k</font>, padding=1)</font>

- _<font color='purple'>pooling layers</font>_ </br>
    This layer dramastically reduce spatial size and computation. We use nn.MaxPool2d(k, k) method in this case.

- _<font color='purple'> Fully Connected Layers</font>_ </br>
    At the end of the CNN, we have fully connected (fc) layers similar to those in MLP. FC layers Combine learned features to classify the image. 
    
    To define a CNN as shown in figure 2, we can use codes like below:</br>
    <font face="Courier New">
        <font color='green'>#convolutional layers </font>
        self.conv1 = nn.Conv2d(3, <font color='blue'>cn1</font>, <font color='blue'>k</font>, padding=1) </br>
        self.conv2 =    
        self.conv3 =    
        <font color='green'>#pooling layer </font> </br>
        self.pool = nn.MaxPool2d(2, 2) </br>
        self.dropout = nn.Dropout(<font color='blue'>p</font>) </br>
        <font color='green'>#fully connected layer </font> </br>
        self.fc1 = nn.Linear(<font color='blue'>X1</font>, <font color='blue'>X2</font>) </br>
        self.fc2 = nn.Linear(<font color='blue'>X2</font>, 10) </br>
    </font>


Similarly, a Relu function should be applied after each convolutional or fc layer to activate the neural network.


**TODO**: Define your own CNN by specifying cn1 (cn2, cn3 ...X1, X2...) , k and p. Build 3 convolutional layers and 2 fc layers. A recommended p value is 0.25, meaning dropping out 25% of features after each fc. You could test to see if dropout is necessary in this case.  </br>

In [ ]:
#Defines a new neural network class inheriting from nn.Module
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # TODO: Define convolutional layers
        # Hint: Start with Conv2d(3, 32, 3), then increase channels

        # TODO end
        
        # Pooling and dropout
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        
        # TODO: Define fully connected layers
        # Hint: After 3 pooling operations, 32x32 becomes 4x4

        # TODO end
        
    def forward(self, x):
        # TODO: Implement the forward pass
        # Hint: Apply conv->relu->pool for each conv layer, then flatten and FC layers

        # TODO end

model = SimpleCNN()
print("CNN Model Structure:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal number of parameters: {total_params:,}")

### 2.4 Define Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1} [Training]', leave=False)
    
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        current_acc = 100 * correct / total
        pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def evaluate(model, test_loader, criterion, epoch):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(test_loader, desc=f'Epoch {epoch+1} [Validation]', leave=False)
    
    with torch.no_grad():
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            current_acc = 100 * correct / total
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

### 2.5 Train the Model

In [ ]:
model = model.to(device)  # Move model to GPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

epochs = 10
print("Starting CNN training...")
print(f"Model will train on: {device}")

# TODO: Implement the training loop with progress tracking
# Hint: Similar to MLP but observe how CNN performs on color images
start_time = time.time()

for epoch in tqdm(range(epochs), desc="CNN Training Progress"):
    epoch_start = time.time()
    
    #define train_loss, train_acc, test_loss, test_acc

    
    epoch_time = time.time() - epoch_start
    
    print(f'Epoch [{epoch+1}/{epochs}] - Time: {epoch_time:.1f}s')
    print(f'  Training   - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%')
    print(f'  Validation - Loss: {test_loss:.4f}, Accuracy: {test_acc:.2f}%')
    print('-' * 50)

total_time = time.time() - start_time
print(f"Training completed on {device}! Total time: {total_time:.1f}s")
# TODO end

### 2.6 Visualize Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, epochs+1), train_losses, 'b-', label='Training Loss', marker='o')
ax1.plot(range(1, epochs+1), test_losses, 'r-', label='Validation Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('CNN Loss During Training')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, epochs+1), train_accuracies, 'b-', label='Training Accuracy', marker='o')
ax2.plot(range(1, epochs+1), test_accuracies, 'r-', label='Validation Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('CNN Accuracy During Training')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Final test accuracy: {test_accuracies[-1]:.2f}%")

### 2.7 Visualize Predictions

In [ ]:
def visualize_cifar_predictions(model, test_loader, classes, num_samples=20):
    model.eval()
    
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    
    # TODO: Handle GPU data transfer for predictions
    images_gpu = images.to(device)
    with torch.no_grad():
        outputs = model(images_gpu)
        _, predictions = torch.max(outputs, 1)
    
    # Move back to CPU for visualization
    images = images.cpu()
    predictions = predictions.cpu()
    # TODO end
    
    # TODO: Visualize CIFAR-10 prediction results
    # Hint: Show denormalized color images with predictions

    # TODO end
    
    plt.suptitle('CIFAR-10 Test Results (Green=Correct, Red=Incorrect)', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    correct = (predictions[:num_samples] == labels[:num_samples]).sum().item()
    accuracy = correct / num_samples * 100
    print(f"Accuracy for displayed samples: {accuracy:.1f}% ({correct}/{num_samples})")

visualize_cifar_predictions(model, test_loader, classes)

---

## Task 3: Fashion-MNIST Classification with ResNet

Fashion-MNIST is a clothing image dataset designed as a direct drop-in replacement for the original MNIST. It maintains the same data format as MNIST: 70,000 grayscale images of 28×28 pixels, divided into 10 classes. The classes include: T-shirt/top, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, and ankle boot. Fashion-MNIST is more challenging than the original MNIST and better suited for evaluating modern deep learning algorithms.

### 3.1 Imports

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

### 3.2 Load Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

print("Downloading Fashion-MNIST dataset...")
train_dataset = torchvision.datasets.FashionMNIST(root='/share/image_cloud/data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root='/share/image_cloud/data', train=False, transform=transform)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

# Fashion-MNIST classes
fashion_classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# TODO: Visualize Fashion-MNIST samples
# Hint: Show one sample from each fashion category

# TODO end

plt.suptitle('Fashion-MNIST Sample Visualization')
plt.tight_layout()
plt.show()

print("\nFashion-MNIST Dataset Information:")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Number of classes: 10")

### 3.3 Define ResNet Model

Residual Network (ResNet) was proposed by Kaiming He et al. in 2015, solving the degradation problem of deep networks by introducing "residual connections". The core innovation of ResNet is the residual block, which allows inputs to be directly passed to outputs through "skip connections", enabling the network to learn residual mappings rather than direct mappings. This design makes it possible to train networks with hundreds or even thousands of layers. ResNet achieved breakthrough performance in the ImageNet competition and has become a foundational architecture for many computer vision tasks.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        # TODO: Define the basic ResNet block
        # Hint: Two conv layers with batch norm, plus shortcut connection


        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        # TODO end
        
    def forward(self, x):
        # TODO: Implement forward pass with residual connection
        # Hint: out = F(x) + x, where F(x) is the residual function

        # TODO end

class SimpleResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleResNet, self).__init__()
        # Initial convolution
        self.conv1 = nn.Conv2d(1, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        
        # TODO: Define residual layers
        # Hint: Use _make_layer to create blocks of residual units

        # TODO end
        
        # Global average pooling and classifier
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels, out_channels, 1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        
        x = self.layer1(x)  # 28x28
        x = self.layer2(x)  # 14x14
        x = self.layer3(x)  # 7x7
        
        x = self.avg_pool(x)  # 1x1
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleResNet()
print("ResNet Model Structure:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal number of parameters: {total_params:,}")

### 3.4 Define Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1} [Training]', leave=False)
    
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        current_acc = 100 * correct / total
        pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def evaluate(model, test_loader, criterion, epoch):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(test_loader, desc=f'Epoch {epoch+1} [Validation]', leave=False)
    
    with torch.no_grad():
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            current_acc = 100 * correct / total
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

### 3.5 Train the Model

In [ ]:
model = model.to(device)  # Move model to GPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

epochs = 10
print("Starting ResNet training...")
print(f"Model will train on: {device}")
print("Observe how ResNet performs compared to simple CNN!")

# TODO: Implement training loop for ResNet with progress tracking
# Hint: Same structure as previous models, observe the performance
start_time = time.time()
#epoch loop


total_time = time.time() - start_time
print(f"Training completed on {device}! Total time: {total_time:.1f}s")
# TODO end

### 3.6 Visualize Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(range(1, epochs+1), train_losses, 'b-', label='Training Loss', marker='o')
ax1.plot(range(1, epochs+1), test_losses, 'r-', label='Validation Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('ResNet Loss During Training')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, epochs+1), train_accuracies, 'b-', label='Training Accuracy', marker='o')
ax2.plot(range(1, epochs+1), test_accuracies, 'r-', label='Validation Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('ResNet Accuracy During Training')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Final test accuracy: {test_accuracies[-1]:.2f}%")

### 3.7 Visualize Predictions

In [ ]:
def visualize_fashion_predictions(model, test_loader, classes, num_samples=20):
    model.eval()
    
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    
    images_gpu = images.to(device)
    with torch.no_grad():
        outputs = model(images_gpu)
        _, predictions = torch.max(outputs, 1)
    
    # Move back to CPU for visualization
    images = images.cpu()
    predictions = predictions.cpu()
    
    # TODO: Visualize Fashion-MNIST predictions
    # Hint: Show grayscale images with fashion class predictions

    # TODO end
    
    plt.suptitle('Fashion-MNIST ResNet Test Results', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    correct = (predictions[:num_samples] == labels[:num_samples]).sum().item()
    accuracy = correct / num_samples * 100
    print(f"Accuracy for displayed samples: {accuracy:.1f}% ({correct}/{num_samples})")

visualize_fashion_predictions(model, test_loader, fashion_classes)

---

## Task 4: Oxford Pets Segmentation with U-Net

The Oxford Pets dataset (officially Oxford-IIIT Pet Dataset) contains 37 pet categories, with 25 dog breeds and 12 cat breeds. Each category has approximately 200 images, totaling about 7,400 images. The dataset provides not only category labels but also detailed annotations, including bounding boxes for pet heads and pixel-level foreground/background segmentation masks. This dataset is commonly used for fine-grained image classification, object detection, and image segmentation tasks.

### 4.1 Imports

In [ ]:
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import OxfordIIITPet
from tqdm.notebook import tqdm

# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

print("Difference between classification and segmentation:")
print("- Classification: Output a class label for the entire image")
print("- Segmentation: Output a class label for each pixel")

### 4.2 Load Data

In [ ]:
# TODO: Define custom dataset class for pet segmentation
# Hint: Need to process the trimap to binary mask

# TODO end

def get_transforms():
    # Image transforms
    image_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Mask transforms
    mask_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])
    
    return image_transform, mask_transform

# Create data loaders
print("Loading Oxford Pet dataset...")
image_transform, mask_transform = get_transforms()

train_dataset = PetSegmentationDataset(
    root='/share/image_cloud/data',
    split='trainval',
    transform=image_transform,
    target_transform=mask_transform
)

test_dataset = PetSegmentationDataset(
    root='/share/image_cloud/data',
    split='test',
    transform=image_transform,
    target_transform=mask_transform
)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

# Create data loaders (small batch size for CPU)
batch_size = 16 if device.type == 'cuda' else 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# TODO: Visualize segmentation data
# Hint: Show original image and binary mask side by side


# TODO end

print("\nOxford Pet Binary Segmentation Dataset Information:")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Mask shape: {train_dataset[0][1].shape}")

### 4.3 Define U-Net Model

U-Net was originally proposed by Ronneberger et al. in 2015, specifically designed for biomedical image segmentation. Its architecture features a symmetric U-shaped structure, consisting of a contracting path (encoder) and an expanding path (decoder). The contracting path progressively extracts high-level features through convolution and pooling operations, while the expanding path recovers spatial resolution through upsampling. The key innovation of U-Net is skip connections, which directly pass feature maps from the encoder to corresponding layers in the decoder, preserving precise localization information. This enables U-Net to achieve accurate pixel-level segmentation even on small datasets.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        # TODO: Define double convolution block
        # Hint: Conv2d -> BatchNorm2d -> ReLU -> Conv2d -> BatchNorm2d -> ReLU

        # TODO end
    
    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        # TODO: Define U-Net architecture with encoder and decoder
        # Hint: Use ModuleList for downs and ups, MaxPool2d for pooling

        # Down part (encoder)

        # Up part (decoder)
 
        # TODO end
    
    def forward(self, x):
        # TODO: Implement U-Net forward pass with skip connections
        # Hint: Encoder -> bottleneck -> Decoder with concatenations
        skip_connections = []
        
        # Encoder
        
        # Decoder
      
        # TODO end

# TODO: Define advanced loss functions for segmentation
# Hint: Combine BCE loss with Dice loss for better performance
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = pred.view(-1)
        target = target.view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.bce = nn.BCELoss()
        self.dice = DiceLoss()
    
    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        dice_loss = self.dice(pred, target)
        return self.alpha * bce_loss + (1 - self.alpha) * dice_loss
# TODO end

# Create model instance
model = UNet(in_channels=3, out_channels=1).to(device)
print("U-Net Model Structure:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal number of parameters: {total_params:,}")

### 4.4 Define Training and Evaluation Functions

In [ ]:
def train_one_epoch_segmentation(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1} [Training]', leave=False)
    
    for data, target in pbar:
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Update progress bar
        pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

def evaluate_segmentation(model, test_loader, criterion, epoch):
    model.eval()
    total_loss = 0
    
    pbar = tqdm(test_loader, desc=f'Epoch {epoch+1} [Validation]', leave=False)
    
    with torch.no_grad():
        for data, target in pbar:
            data = data.to(device)
            target = target.to(device)
            
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            
            # Update progress bar
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(test_loader)
    return avg_loss

### 4.5 Train the Model

In [ ]:
# TODO: Setup Combined Loss for binary segmentation
criterion =
# TODO end

def save_checkpoint(model, optimizer, epoch, train_losses, test_losses, 
                   best_loss, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'test_losses': test_losses,
        'best_loss': best_loss,
    }
    
    torch.save(checkpoint, './task4_unet_latest.pth')
    print(f"Checkpoint saved: task4_unet_latest.pth")
    
    if is_best:
        torch.save(checkpoint, './task4_unet_best.pth')
        print(f"Best model saved: task4_unet_best.pth")

def load_checkpoint(model, optimizer, checkpoint_path):
    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        train_losses = checkpoint['train_losses']
        test_losses = checkpoint['test_losses']
        best_loss = checkpoint['best_loss']
        
        print(f"Resuming from epoch {start_epoch}")
        print(f"Best loss so far: {best_loss:.4f}")
        
        return start_epoch, train_losses, test_losses, best_loss
    else:
        print("No checkpoint found, starting from scratch")
        return 0, [], [], float('inf')

# TODO: Setup optimizer for U-Net
optimizer = 
# TODO end

# Load checkpoint if exists
checkpoint_path = "./task4_unet_latest.pth"
start_epoch, train_losses, test_losses, best_loss = load_checkpoint(
    model, optimizer, checkpoint_path
)

total_epochs = 10
remaining_epochs = total_epochs - start_epoch

if remaining_epochs <= 0:
    print("Binary segmentation training already completed!")
else:
    print(f"Starting binary segmentation training from epoch {start_epoch+1}/{total_epochs}")
    print("Task: Pet foreground vs background segmentation")
    
    # TODO: Binary segmentation training loop
    start_time = time.time()
    
    for epoch in tqdm(range(start_epoch, total_epochs), desc="Binary Segmentation Training"):
        epoch_start = time.time()
        
        train_loss = train_one_epoch_segmentation(model, train_loader, criterion, optimizer, epoch)
        test_loss = evaluate_segmentation(model, test_loader, criterion, epoch)
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        epoch_time = time.time() - epoch_start
        
        # Check if best model
        is_best = test_loss < best_loss
        if is_best:
            best_loss = test_loss
        
        # Save checkpoint
        save_checkpoint(model, optimizer, epoch, train_losses, test_losses, 
                       best_loss, is_best)
        
        print(f'Epoch [{epoch+1}/{total_epochs}] - Time: {epoch_time:.1f}s')
        print(f'  Training   - Loss: {train_loss:.4f}')
        print(f'  Validation - Loss: {test_loss:.4f}')
        if is_best:
            print(f'  *** NEW BEST BINARY MODEL! Loss: {test_loss:.4f} ***')
        print('-' * 50)
    
    total_time = time.time() - start_time
    print(f"Binary segmentation training completed! Total time: {total_time:.1f}s")
    print(f"Best validation loss achieved: {best_loss:.4f}")
    # TODO end

print(f"\nBinary Segmentation Training Summary:")
print(f"• Task: Pet foreground vs background")
print(f"• Loss: Combined Dice + BCE")
print(f"• Best loss: {min(test_losses) if test_losses else float('inf'):.4f}")
print(f"• Model output: Single channel probability map")

### 4.6 Visualize Training Results

In [ ]:
# TODO: Load best binary model for evaluation
print("Loading best binary segmentation model...")
if os.path.exists('./task4_unet_best.pth'):
    checkpoint = torch.load('./task4_unet_best.pth', map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model with loss: {checkpoint['best_loss']:.4f}")
else:
    print("Using current model (no best checkpoint found)")
# TODO end

# Visualize training progress
epochs_trained = len(train_losses)
if epochs_trained > 0:
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # TODO: Plot training curves for segmentation

    # TODO end
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final validation loss: {test_losses[-1]:.4f}")
    print(f"Best validation loss: {min(test_losses):.4f}")
else:
    print("No training history to visualize")

### 4.7 Visualize Predictions

In [ ]:
def visualize_predictions(model, test_loader, threshold=0.5, num_samples=8):
    model.eval()
    dataiter = iter(test_loader)
    images, true_masks = next(dataiter)
    
    with torch.no_grad():
        images_gpu = images.to(device)
        pred_masks = model(images_gpu).cpu()
    
    # TODO: Visualize segmentation predictions
    # Hint: Show original image, ground truth mask, and predicted mask
    fig, axes = plt.subplots(3, num_samples, figsize=(15, 9))
    
    for i in range(num_samples):
        # Denormalize image for display
        img = images[i].cpu()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img * std + mean
        img = torch.clamp(img, 0, 1)

        if threshold is not None and 0 < threshold < 1:
            # Apply threshold to predicted mask if specified
            pred_masks[i] = (pred_masks[i] > threshold).float()
        
        # Original image

        
        # Ground truth mask

        
        # Predicted mask

    # TODO end
    
    plt.suptitle('U-Net Pet Segmentation Results\n(White=Foreground, Black=Background)', fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize segmentation results
visualize_predictions(model, test_loader)

---

## Task 5: Improved Oxford Pets Segmentation with ResNet and U-Net

The Oxford Pets dataset (officially Oxford-IIIT Pet Dataset) contains 37 pet categories, with 25 dog breeds and 12 cat breeds. Each category has approximately 200 images, totaling about 7,400 images. The dataset provides not only category labels but also detailed annotations, including bounding boxes for pet heads and pixel-level foreground/background segmentation masks. This dataset is commonly used for fine-grained image classification, object detection, and image segmentation tasks.

### 5.1 Imports

In [ ]:
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import OxfordIIITPet
from torchvision.models.resnet import ResNet18_Weights
from tqdm.notebook import tqdm

torch.manual_seed(42)
np.random.seed(42)

# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

### 5.2 Load Data

In [ ]:
class PetSegmentationDataset(Dataset):
    def __init__(self, root, split='trainval', transform=None, target_transform=None):
        # Download Oxford Pet dataset
        self.pet_dataset = OxfordIIITPet(
            root=root, 
            split=split, 
            target_types='segmentation',
            download=True
        )
        self.transform = transform
        self.target_transform = target_transform
    
    def __len__(self):
        return len(self.pet_dataset)
    
    def __getitem__(self, idx):
        image, target = self.pet_dataset[idx]
        
        # Convert target to binary mask (focus on foreground)
        # Oxford Pet trimap: 1=foreground, 2=background, 3=boundary/unclassified
        target = np.array(target)
        # Set foreground to 1, others to 0
        mask = (target == 1).astype(np.float32)
        mask = Image.fromarray(mask)
        
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
            
        return image, mask

def get_transforms():
    # Image transforms
    image_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Mask transforms
    mask_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])
    
    return image_transform, mask_transform

# Create data loaders
print("Loading Oxford Pet dataset...")
image_transform, mask_transform = get_transforms()

train_dataset = PetSegmentationDataset(
    root='./data',
    split='trainval',
    transform=image_transform,
    target_transform=mask_transform
)

test_dataset = PetSegmentationDataset(
    root='./data',
    split='test',
    transform=image_transform,
    target_transform=mask_transform
)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

batch_size = 16 if device.type == 'cuda' else 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(f"Using batch size: {batch_size}")

def visualize_data(data_loader, num_samples=5):
    dataiter = iter(data_loader)
    images, masks = next(dataiter)
    
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
    
    for i in range(num_samples):
        # Denormalize image for display
        img = images[i].cpu()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        
        # Show original image
        axes[0, i].imshow(img.permute(1, 2, 0))
        axes[0, i].set_title('Original Image')
        axes[0, i].axis('off')
        
        # Show mask
        mask = masks[i].cpu().squeeze()
        axes[1, i].imshow(mask, cmap='gray')
        axes[1, i].set_title('Foreground Mask')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_data(train_loader)

print("\nOxford Pet Binary Segmentation Dataset Information:")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Mask shape: {train_dataset[0][1].shape}")

### 5.3 Define ResNet U-Net Model

Use ResNet as the encoder for U-Net, leveraging its powerful feature extraction capabilities. The ResNet encoder will extract high-level features, while the U-Net decoder will reconstruct the segmentation map.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        
        # TODO: Load pretrained ResNet as encoder for binary segmentation
        # Hint: Use ResNet18 backbone and extract feature layers
        resnet = torchvision.models.resnet18(weights=ResNet18_Weights.DEFAULT)
        
        # Extract encoder layers
        self.encoder1 = nn.Sequential(
            resnet.conv1,      # 3->64, stride=2, 256->128
            resnet.bn1,
            resnet.relu
        )
        self.encoder2 = nn.Sequential(
            resnet.maxpool,    # stride=2, 128->64
            resnet.layer1      # 64->64
        )
        self.encoder3 = resnet.layer2  # 64->128, stride=2, 64->32
        self.encoder4 = resnet.layer3  # 128->256, stride=2, 32->16
        self.encoder5 = resnet.layer4  # 256->512, stride=2, 16->8
        # TODO end
        
        # TODO: Define decoder for binary segmentation
        # Hint: Single output channel for binary mask
        # Decoder

        
        # Binary output
        self.outc =        # Single channel for binary segmentation
        self.sigmoid = 
        # TODO end
        
        # Freeze early encoder layers to prevent overfitting
        for param in self.encoder1.parameters():
            param.requires_grad = False
        for param in self.encoder2.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # TODO: Implement forward pass for binary segmentation
        # Hint: Encoder-decoder with skip connections, single output channel
        # Encoder path

        
        # Decoder path with skip connections

        
        # Binary output with sigmoid activation
        return logits, probs
        # TODO end

class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = pred.view(-1)
        target = target.view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice

class CombinedLoss(nn.Module):
    """Combined Dice + BCE Loss for binary segmentation"""
    def __init__(self, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.bce = nn.BCELoss()
        self.dice = DiceLoss()
    
    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        dice_loss = self.dice(pred, target)
        return self.alpha * bce_loss + (1 - self.alpha) * dice_loss

# Create model and move to device
model = UNet().to(device)
print("U-Net Model Structure:")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {frozen_params:,}")
print(f"Model moved to: {device}")

### 5.4 Define Training and Evaluation Functions

In [ ]:
def train_one_epoch_binary(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct_pixels = 0
    total_pixels = 0
    
    pbar = tqdm(enumerate(train_loader), desc=f'Epoch {epoch+1} [Training]', 
                total=len(train_loader), leave=True)
    
    for batch_idx, (data, target) in pbar:
        
        data, target = data.to(device), target.to(device)
        target = target.squeeze(1)  # Remove channel dimension for binary mask
        
        optimizer.zero_grad()
        logits, probs = model(data)
        
        # TODO: Use probabilities for Combined Loss
        loss = 
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate pixel accuracy (threshold at 0.5)
        predicted = (probs.squeeze(1) > 0.5).float()
        total_pixels += target.numel()
        correct_pixels += (predicted == target).sum().item()
        
        current_acc = 100 * correct_pixels / total_pixels
        pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'PixelAcc': f'{current_acc:.2f}%'})
    
    avg_loss = total_loss / len(train_loader)
    pixel_accuracy = 100 * correct_pixels / total_pixels
    return avg_loss, pixel_accuracy

def evaluate_binary(model, test_loader, criterion, epoch):
    model.eval()
    total_loss = 0
    correct_pixels = 0
    total_pixels = 0
    
    pbar = tqdm(enumerate(test_loader), desc=f'Epoch {epoch+1} [Validation]', 
                total=len(test_loader), leave=True)
    
    with torch.no_grad():
        for batch_idx, (data, target) in pbar:
            
            data, target = data.to(device), target.to(device)
            target = target.squeeze(1)
            
            logits, probs = model(data)
            loss = criterion(probs.squeeze(1), target)
            
            total_loss += loss.item()
            
            predicted = (probs.squeeze(1) > 0.5).float()
            total_pixels += target.numel()
            correct_pixels += (predicted == target).sum().item()
            
            current_acc = 100 * correct_pixels / total_pixels
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'PixelAcc': f'{current_acc:.2f}%'})
    
    avg_loss = total_loss / len(test_loader)
    pixel_accuracy = 100 * correct_pixels / total_pixels
    return avg_loss, pixel_accuracy

### 5.5 Train the Model

In [ ]:
criterion = CombinedLoss(alpha=0.5)

def save_checkpoint(model, optimizer, epoch, train_losses, train_accuracies, 
                   test_losses, test_accuracies, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies,
        'best_accuracy': max(test_accuracies) if test_accuracies else 0,
    }
    
    torch.save(checkpoint, './task4_resunet_latest.pth')
    print(f"Checkpoint saved: task4_resunet_latest.pth")
    
    if is_best:
        torch.save(checkpoint, './task4_resunet_best.pth')
        print(f"Best model saved: task4_resunet_best.pth")

def load_checkpoint(model, optimizer, checkpoint_path):
    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        train_losses = checkpoint['train_losses']
        train_accuracies = checkpoint['train_accuracies']
        test_losses = checkpoint['test_losses']
        test_accuracies = checkpoint['test_accuracies']
        best_accuracy = checkpoint['best_accuracy']
        
        print(f"Resuming from epoch {start_epoch}")
        print(f"Best accuracy so far: {best_accuracy:.2f}%")
        
        return start_epoch, train_losses, train_accuracies, test_losses, test_accuracies, best_accuracy
    else:
        print("No checkpoint found, starting from scratch")
        return 0, [], [], [], [], 0

# TODO: Setup optimizer with different learning rates for binary task
encoder_params = []
decoder_params = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if 'encoder' in name:
            encoder_params.append(param)
        else:
            decoder_params.append(param)

optimizer = optim.Adam([
    {'params': encoder_params, 'lr': 0.0001},  # Lower for pretrained
    {'params': decoder_params, 'lr': 0.001}    # Higher for new decoder
])
print("Using optimized learning rates for binary segmentation")
# TODO end

# Load checkpoint if exists
start_epoch, train_losses, train_accuracies, test_losses, test_accuracies, best_accuracy = load_checkpoint(
    model, optimizer, './task4_resunet_latest.pth'
)

total_epochs = 10
remaining_epochs = total_epochs - start_epoch

if remaining_epochs <= 0:
    print("Binary segmentation training already completed!")
else:
    print(f"Starting binary segmentation training from epoch {start_epoch+1}/{total_epochs}")
    print("Task: Pet foreground vs background segmentation")
    
    # TODO: Binary segmentation training loop
    start_time = time.time()
    
    for epoch in tqdm(range(start_epoch, total_epochs), desc="Binary Segmentation Training"):
        epoch_start = time.time()
        
        train_loss, train_acc = train_one_epoch_binary(model, train_loader, criterion, optimizer, epoch)
        test_loss, test_acc = evaluate_binary(model, test_loader, criterion, epoch)
        
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        test_losses.append(test_loss)
        test_accuracies.append(test_acc)
        
        epoch_time = time.time() - epoch_start
        
        # Check if best model
        is_best = test_acc > best_accuracy
        if is_best:
            best_accuracy = test_acc
        
        # Save checkpoint
        save_checkpoint(model, optimizer, epoch, train_losses, train_accuracies, 
                       test_losses, test_accuracies, is_best)
        
        print(f'Epoch [{epoch+1}/{total_epochs}] - Time: {epoch_time:.1f}s')
        print(f'  Training   - Loss: {train_loss:.4f}, Pixel Accuracy: {train_acc:.2f}%')
        print(f'  Validation - Loss: {test_loss:.4f}, Pixel Accuracy: {test_acc:.2f}%')
        if is_best:
            print(f'  *** NEW BEST BINARY MODEL! Accuracy: {test_acc:.2f}% ***')
        print('-' * 50)
    
    total_time = time.time() - start_time
    print(f"Binary segmentation training completed! Total time: {total_time:.1f}s")
    print(f"Best pixel accuracy achieved: {best_accuracy:.2f}%")
    # TODO end

print(f"\nBinary Segmentation Training Summary:")
print(f"• Task: Pet foreground vs background")
print(f"• Loss: Combined Dice + BCE")
print(f"• Best accuracy: {max(test_accuracies) if test_accuracies else 0:.2f}%")
print(f"• Model output: Single channel probability map")

### 5.6 Visualize Training Results

In [ ]:
print("Loading best binary segmentation model...")
if os.path.exists('./task4_resunet_best.pth'):
    checkpoint = torch.load('./task4_resunet_best.pth', map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model with accuracy: {checkpoint['best_accuracy']:.2f}%")
else:
    print("Using current model (no best checkpoint found)")

# Visualize training progress
epochs_trained = len(train_losses)
if epochs_trained > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(range(1, epochs_trained+1), train_losses, 'b-', label='Training Loss', marker='o')
    ax1.plot(range(1, epochs_trained+1), test_losses, 'r-', label='Validation Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Combined Loss')
    ax1.set_title('Binary Segmentation Loss During Training')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(range(1, epochs_trained+1), train_accuracies, 'b-', label='Training Accuracy', marker='o')
    ax2.plot(range(1, epochs_trained+1), test_accuracies, 'r-', label='Validation Accuracy', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Pixel Accuracy (%)')
    ax2.set_title('Binary Segmentation Accuracy During Training')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Final pixel accuracy: {test_accuracies[-1]:.2f}%")
else:
    print("No training history to visualize")

### 5.7 Visualize Predictions

In [ ]:
def visualize_predictions(model, test_loader, threshold=0.5, num_samples=8):
    """Visualize segmentation predictions"""
    model.eval()
    dataiter = iter(test_loader)
    images, true_masks = next(dataiter)
    
    with torch.no_grad():
        images_gpu = images.to(device)
        logits, probs = model(images_gpu)
        pred_masks = probs.cpu()
    
    fig, axes = plt.subplots(4, num_samples, figsize=(15, 12))
    
    for i in range(num_samples):
        # Denormalize image for display
        img = images[i].cpu()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        
        # Get probability map and binary prediction
        prob_map = pred_masks[i].squeeze()
        binary_pred = (prob_map > threshold).float()
        
        # Original image
        axes[0, i].imshow(img.permute(1, 2, 0))
        axes[0, i].set_title('Original Image')
        axes[0, i].axis('off')
        
        # Ground truth mask
        true_mask = true_masks[i].squeeze()
        axes[1, i].imshow(true_mask, cmap='gray', vmin=0, vmax=1)
        axes[1, i].set_title('Ground Truth')
        axes[1, i].axis('off')
        
        # Probability map
        axes[2, i].imshow(prob_map, cmap='jet', vmin=0, vmax=1)
        axes[2, i].set_title('Probability Map')
        axes[2, i].axis('off')
        
        # Binary prediction
        axes[3, i].imshow(binary_pred, cmap='gray', vmin=0, vmax=1)
        axes[3, i].set_title(f'Prediction (τ={threshold})')
        axes[3, i].axis('off')
    
    plt.suptitle('ResUNet Binary Pet Segmentation Results\n(White=Foreground, Black=Background)', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    print("Binary Segmentation Results Analysis:")
    print("- Probability map shows model confidence (red=high, blue=low)")
    print("- Binary mask generated using threshold τ=0.5")
    print("- White regions represent detected pet foreground")
    print("- ResNet encoder provides strong feature extraction")

def calculate_metrics(model, test_loader, num_batches=50):
    model.eval()
    total_dice = 0
    total_iou = 0
    total_accuracy = 0
    count = 0
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            if batch_idx >= num_batches:
                break
                
            data, target = data.to(device), target.to(device)
            target = target.squeeze(1)
            
            _, probs = model(data)
            preds = (probs.squeeze(1) > 0.5).float()
            
            # Calculate metrics for each image in batch
            for i in range(data.size(0)):
                pred = preds[i].flatten()
                tgt = target[i].flatten()
                
                # Dice coefficient
                intersection = (pred * tgt).sum()
                dice = (2. * intersection + 1e-7) / (pred.sum() + tgt.sum() + 1e-7)
                
                # IoU (Jaccard)
                union = pred.sum() + tgt.sum() - intersection
                iou = (intersection + 1e-7) / (union + 1e-7)
                
                # Pixel accuracy
                acc = (pred == tgt).float().mean()
                
                total_dice += dice.item()
                total_iou += iou.item()
                total_accuracy += acc.item()
                count += 1
    
    avg_dice = total_dice / count
    avg_iou = total_iou / count
    avg_acc = total_accuracy / count
    
    print(f"\nBinary Segmentation Metrics (on {count} samples):")
    print(f"- Dice Coefficient: {avg_dice:.4f}")
    print(f"- IoU (Jaccard): {avg_iou:.4f}")  
    print(f"- Pixel Accuracy: {avg_acc:.4f}")
    
    return avg_dice, avg_iou, avg_acc

# Visualize segmentation results
visualize_predictions(model, test_loader)

# Calculate and display metrics
print("\nCalculating comprehensive binary segmentation metrics...")
dice_score, iou_score, pixel_acc = calculate_metrics(model, test_loader)

## Task 6: Compare CPU and GPU Performance

Choose one task from task 1~5 and run again on GPU. Open the gpu-version of lab2 notebook and connect to GPU. Record the total training time.
- Plot the time needed for CPU and GPU configurations. </br>
- Plot a total cost vs machine configuration bar chart. (optional)

Recomended comparison: 
| Machine Type       | CPU      | CPU       | GPU          | GPU                     |
|--------------------|----------|-----------|--------------|-------------------------|
| Machine Configuration | c2m4     | c4m16     |c3m4* NVIDIA T4    | c8_m32_1 * NVIDIA V100 <br> (optional) |
| Cost (¥/h)         | 0        | 0.252     | 0.66         | 11                      |
| training time (s)      | ?        | ?    | ?       | ?                     |

In [ ]:
#TODO: Generate a barchart plot for training time comparison
#x-axis represents machine configuration (e.g., CPU(c2m4), GPU(NVIDIA T4)
#y-axis represents training time in seconds

#TODO: end

#TODO: Plot machine cost comparison (optional)
#x-axis represents machine type (e.g., CPU(c2m4), GPU(NVIDIA T4)
#y-axis represents total cost in yuan


#TODO: end

Notebook created by: [Zehua Zhao 赵泽华 zzh@stu.pku.edu.cn](mailto:zzh@stu.pku.edu.cn)  <br>
*Assisted by: Claude Opus 4* <br>
Slightly modified by: Xiao Li

---